In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import torch

In [ ]:
model_id = "google/gemma-3-12b-it-qat-q4_0-gguf"  # Base model 

: 

In [ ]:
data = load_dataset('csv', data_files='elderly_chat.csv')

In [ ]:
def format_chat(example):
    prompt = f"<start_of_turn>user\n{example['instruction']}<end_of_turn>\n<start_of_turn>model\n{example['response']}<end_of_turn>"
    return {"text": prompt}
data = data.map(format_chat)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_4bit=True,
    device_map="auto"
)

In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=512)

tokenized_data = data['train'].map(tokenize, remove_columns=data['train'].column_names)

In [ ]:
training_args = TrainingArguments(
    output_dir="./gemma-chat-elderly",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    fp16=True,
    save_strategy="epoch",
    logging_dir="./logs",
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    tokenizer=tokenizer
)

In [ ]:
trainer.train()
output_dir="./finetuned-model"
model.save_pretrained("output_dir")
tokenizer.save_pretrained("output_dir")
print(f"Model saved to {output_dir}")